![image](https://github.com/user-attachments/assets/b22c9807-f5e7-49eb-b00d-598e400781af)

# Agno Agents with GPT-OSS 120B

## Introduction

Agno is a framework for building AI agents that can use tools, access knowledge bases, and work together in teams. This notebook demonstrates how to use Agno with [GPT-OSS 120B](https://clarifai.com/openai/chat-completion/models/gpt-oss-120b) model from the Clarifai Community through OpenAI-compatible API to create powerful AI Agents with different capabilities.

In this notebook, we'll explore:

1. **Setting up a simple agent** with web search capability using GPT-OSS 120B
2. **Creating an agent with a knowledge base** for specialized domains
3. **Building multi-agent systems** where specialized agents work together


## Install Required Packages

Install required packages for this notebook:
- `agno`: The agent framework we'll be using
- `duckduckgo-search`: For web search capabilities  
- `pypdf`: For processing PDF documents
- `lancedb` and `tantivy`: For vector storage
- `yfinance`: For financial data retrieval


In [ ]:
!pip install -qU agno duckduckgo-search pypdf lancedb tantivy yfinance openai ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.5/289.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/948.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.5/301.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42

### Setup your PAT key
Set your Clarifai PAT as environment variable.
Below we will be using Clarifai PAT as alias for OPENAI API KEY, since we are using Clarifai models in OpenAI compatible endpoints format.

In [ ]:
!export CLARIFAI_PAT="YOUR_CLARIFAI_PAT"  # Set your Clarifai PAT here

### Clarifai LLM model

Agno AGI supports the OpenAI-compatible format for calling LLM models. Models available on the Clarifai Community can be accessed using Clarifai’s OpenAI-compatible endpoint by specifying the base URL and model name.  

#### Using Clarifai Models
Clarifai models can be accessed using the OpenAI-compatible format in the following model URL structure. It starts with the prefix `openai` -  

`openai/{user_id}/{app_id}/models/{model_id}`

### Available Models

You can explore available models on the [Clarifai Community](https://clarifai.com/explore) platform. Some popular models include:


- **GPT-OSS-120B** → `openai/chat-completion/models/gpt-oss-120b`  
- **Grok Code Fast 1** → `xai/chat-completion/models/grok-code-fast-1`  
- **DeepSeek V3.1** → `deepseek-ai/deepseek-chat/models/DeepSeek-V3_1`  
- **Kimi K2 Instruct** → `moonshotai/kimi/models/Kimi-K2-Instruct`  
- **GLM 4.5** → `zai/completion/models/GLM_4_5`  


## Simple Agent

Below we create a basic agent that can search the web to answer questions. This agent uses:
- **Clarifai's GPT-OSS 120B model** via OpenAI-compatible API
- **DuckDuckGo search** as a tool to retrieve information from the web

When we run this agent with a question about current events, it will search for recent information and provide an answer.



In [ ]:
import os
from agno.agent import Agent
from agno.models.openai.like import OpenAILike
from agno.tools.duckduckgo import DuckDuckGoTools

# Create agent using Clarifai's GPT-OSS 120B model
agent = Agent(
    model=OpenAILike(
        id="openai/chat-completion/models/gpt-oss-120b",  # Clarifai's GPT-OSS 120B model
        api_key=os.environ["CLARIFAI_PAT"],
        base_url="https://api.clarifai.com/v2/ext/openai/v1",
        max_tokens=4096
    ),
    tools=[DuckDuckGoTools()],
    description="You are a helpful AI assistant powered by Clarifai's GPT-OSS 120B model.",
    markdown=True
)

agent.print_response("How are the golden state warriors doing this 2024-2025 season?", stream=True)


Output()

## Agent with Knowledge Base

This example demonstrates how to create an agent with specialized knowledge. We'll build a Thai cuisine expert by:

1. **Loading a PDF** containing Thai recipes into a knowledge base
2. **Setting up a vector database** to store and retrieve this information efficiently  
3. **Configuring the agent** to prioritize its knowledge base while still being able to search the web

This type of agent is ideal for specialized domains where you want to combine proprietary knowledge with the ability to find supplementary information.

For generating embeddings, we'll be using the `text-embedding-ada-002` model from Clarifai community with an OpenAI-compatible API.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import os
from typing import Optional
from agno.agent import Agent
from agno.models.openai.like import OpenAILike
from agno.knowledge.embedder.openai import OpenAIEmbedder
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.tools.duckduckgo import DuckDuckGoTools

# ---------------------------------------------------------
# Define the embedder (Clarifai OpenAI-like embedding)
embedder = OpenAIEmbedder(
    id="openai/embed/models/text-embedding-ada-002",
    api_key=os.environ["CLARIFAI_PAT"],
    base_url="https://api.clarifai.com/v2/ext/openai/v1"
)

# Initialize LanceDB vector database
vector_db = LanceDb(
    uri="tmp/lancedb",
    table_name="thai_recipes",
    search_type=SearchType.hybrid,
    embedder=embedder,
)

# Load the knowledge base
knowledge_base = Knowledge(vector_db=vector_db)

# Add PDF content to knowledge base
knowledge_base.add_content(
        url="https://agno-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf"
    )

# ---------------------------------------------------------
# Initialize the agent
agent = Agent(
    model=OpenAILike(
        id="openai/chat-completion/models/gpt-oss-120b",
        api_key=os.environ["CLARIFAI_PAT"],
        base_url="https://api.clarifai.com/v2/ext/openai/v1",
        max_tokens=4096
    ),
    description="You are a Thai cuisine expert powered by Clarifai's GPT-OSS 120B!",
    instructions=[
        "Search your knowledge base for Thai recipes first.",
        "If the question requires current information, search the web to fill in gaps.",
        "Prefer the information in your knowledge base over web results for recipe-related queries.",
        "Always provide detailed cooking instructions and ingredient lists when available."
    ],
    knowledge=knowledge_base,
    tools=[DuckDuckGoTools()],
    markdown=True
)

# ---------------------------------------------------------
# Example usage
query = "How do I make chicken and galangal in coconut milk soup?"
agent.print_response(query, markdown=True)

INFO Loading content: b9d61209-73e8-5072-8d39-84263a15b9d6

INFO Adding content from URL https://agno-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf

WARNING  Contents DB not found for knowledge base: None

Output()

INFO Found 10 documents

INFO Found 10 documents

In [ ]:
agent.print_response("What is the history of Thai curry and how has it evolved?", stream=True)


Output()

INFO Found 10 documents

INFO Found 10 documents

## Multi-Agent System

This example demonstrates how to create a team of specialized agents that work together. We'll build:

1. **A web search agent** for finding general information
2. **A finance agent** for retrieving financial data  
3. **A coordinator agent** that delegates tasks to the specialized agents

This approach allows us to create more powerful systems by combining specialized capabilities.



In [ ]:
import os
from agno.agent import Agent
from agno.team import Team
from agno.models.openai.like import OpenAILike
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools

# Create individual agents
web_agent = Agent(
    name="Web Research Agent",
    role="Search the web for current information and news",
    model=OpenAILike(
        id="openai/chat-completion/models/gpt-oss-120b",
        api_key=os.environ["CLARIFAI_PAT"],
        base_url="https://api.clarifai.com/v2/ext/openai/v1",
        max_tokens=4096,
    ),
    tools=[DuckDuckGoTools()],
    instructions=[
        "Always include sources in your responses",
        "Focus on finding recent and relevant information",
        "Provide comprehensive summaries with key details",
    ],
    markdown=True,
)

finance_agent = Agent(
    name="Financial Analysis Agent",
    role="Retrieve and analyze financial data",
    model=OpenAILike(
        id="openai/chat-completion/models/gpt-oss-120b",
        api_key=os.environ["CLARIFAI_PAT"],
        base_url="https://api.clarifai.com/v2/ext/openai/v1",
        max_tokens=4096,
    ),
    tools=[YFinanceTools()],
    instructions=[
        "Use tables to display financial data clearly",
        "Provide context and analysis with raw numbers",
        "Include relevant financial metrics and ratios",
        "Explain trends and their potential implications",
    ],
    markdown=True,
)

# Create a team with the agents as members
agent_team = Team(
    members=[web_agent, finance_agent],
    model=OpenAILike(
        id="openai/chat-completion/models/gpt-oss-120b",
        api_key=os.environ["CLARIFAI_PAT"],
        base_url="https://api.clarifai.com/v2/ext/openai/v1",
        max_tokens=4096,
    ),
    instructions=[
        "Coordinate between research and financial analysis tasks",
        "Always include sources and cite data properly",
        "Use tables and structured formatting for clarity",
        "Provide comprehensive analysis combining both web research and financial data",
        "Synthesize information from multiple sources into coherent insights",
    ],
    markdown=True,
)

# Example usage
agent_team.print_response(
    "What's the current market outlook and financial performance of major AI semiconductor companies like NVIDIA, AMD, and Intel?",
    stream=True,
)

Output()

In this notebook, we've demonstrated several approaches to building AI agents with Agno and Clarifai's GPT-OSS 120B model:

### **1. Simple Agent**
- Quick to set up and useful for general tasks requiring web search
- Direct integration with Clarifai's OpenAI-compatible API

### **2. Knowledge Base Agent**
- Combines specialized knowledge with web search capabilities
- Perfect for domain-specific applications with proprietary data

### **3. Multi-Agent System**
- Coordinates specialized agents for complex tasks requiring different capabilities
- Demonstrates team-based AI problem solving

### **Next Steps:**

You can extend these examples by:
- **Adding more specialized tools** for your specific domain
- **Building larger knowledge bases** with domain-specific information  
- **Creating more complex agent teams** with sophisticated coordination patterns
- **Integrating with other Clarifai services** like visual models for multimodal capabilities
- **Implementing persistent memory** for agents that learn from interactions

